# 13 — Which candidates break the money → support pattern? (2024)

This is the **case-finding notebook**.

It asks three concrete questions:

1. Which candidates receive substantially more or fewer mentions than a
   simple finance model predicts?
2. Which same-district candidates raised very similar amounts but received
   very different support?
3. Which candidates have broad support versus deep first-choice support?

The notebook identifies cases. It does **not** assign political explanations.
Those explanations must come from sourced bios, endorsements, campaign history,
or other qualitative evidence.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 100)

# Find the repository root from either the repo root or notebooks/.
cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


## 2. Load the top-line candidate table

In [2]:
topline_path = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "topline"
    / "candidate_topline_finance_support.csv"
)

analysis = pd.read_csv(
    topline_path
)

print("Candidates:", len(analysis))
print(
    "With fundraising:",
    analysis["fundraising"].notna().sum(),
)


Candidates: 98
With fundraising: 65


## 3. Residuals: who does better or worse than fundraising predicts?

**Important correction from the earlier draft:** Notebook 10 found that raw
fundraising dollars fit mentions better than log fundraising.

Therefore the residual model here uses:

`mentions = a + b × raw fundraising`

separately inside each district.

Residual = actual mentions − predicted mentions.


In [3]:
def district_residuals(data):
    rows = []

    for district in sorted(
        data["district"].dropna().unique()
    ):
        district_data = data.loc[
            data["district"].eq(district),
            [
                "candidate_key",
                "canonical_candidate",
                "district",
                "is_viable",
                "fundraising",
                "contribution_count",
                "mentions",
                "first_place_votes",
            ],
        ].dropna(
            subset=[
                "fundraising",
                "mentions",
            ]
        ).copy()

        if len(district_data) < 4:
            continue

        x = district_data[
            "fundraising"
        ].to_numpy(dtype=float)

        y = district_data[
            "mentions"
        ].to_numpy(dtype=float)

        model = sm.OLS(
            y,
            sm.add_constant(x),
        ).fit()

        district_data["predicted_mentions"] = (
            model.predict(
                sm.add_constant(x)
            )
        )

        district_data["residual"] = (
            district_data["mentions"]
            - district_data["predicted_mentions"]
        )

        district_data["district_r_squared"] = (
            model.rsquared
        )

        # Basic diagnostic: a negative predicted mention count is impossible.
        district_data["impossible_prediction"] = (
            district_data["predicted_mentions"]
            < 0
        )

        # Simple influence diagnostic:
        # refit the line without each candidate and see how much the slope changes.
        slope_changes = []

        for position in range(
            len(district_data)
        ):
            keep = np.ones(
                len(district_data),
                dtype=bool,
            )

            keep[position] = False

            refit = sm.OLS(
                y[keep],
                sm.add_constant(
                    x[keep]
                ),
            ).fit()

            original_slope = (
                model.params[1]
            )

            if original_slope == 0:
                slope_change = np.nan
            else:
                slope_change = (
                    abs(
                        refit.params[1]
                        - original_slope
                    )
                    / abs(original_slope)
                )

            slope_changes.append(
                slope_change
            )

        district_data[
            "slope_change_if_dropped"
        ] = slope_changes

        rows.append(
            district_data
        )

    return pd.concat(
        rows,
        ignore_index=True,
    )


residuals = district_residuals(
    analysis
)

residuals["absolute_residual"] = (
    residuals["residual"].abs()
)

print("Residuals computed:", len(residuals))
print(
    "Impossible predictions:",
    residuals["impossible_prediction"].sum(),
)


Residuals computed: 65
Impossible predictions: 0


In [4]:
largest_residuals = (
    residuals
    .sort_values(
        "absolute_residual",
        ascending=False,
    )
    .head(16)
)

display(
    largest_residuals[
        [
            "district",
            "canonical_candidate",
            "fundraising",
            "mentions",
            "predicted_mentions",
            "residual",
            "absolute_residual",
            "slope_change_if_dropped",
            "impossible_prediction",
        ]
    ]
    .round(
        {
            "fundraising": 0,
            "mentions": 0,
            "predicted_mentions": 0,
            "residual": 0,
            "slope_change_if_dropped": 3,
        }
    )
)


,district,canonical_candidate,fundraising,mentions,predicted_mentions,residual,absolute_residual,slope_change_if_dropped,impossible_prediction
53,4,Stanley Penkin,61847.0,9570.0,29039.0,-19469.0,19468.908316,0.148,False
57,4,Sarah Silkie,25478.0,32039.0,15236.0,16803.0,16802.509062,0.036,False
31,2,Michelle DePass,32578.0,32612.0,16058.0,16554.0,16553.918836,0.016,False
56,4,Eric Zimmerman,41879.0,36731.0,21461.0,15270.0,15270.253117,0.031,False
25,2,Sameer Kanal,34712.0,31168.0,16660.0,14508.0,14507.988220,0.007,False
19,2,Elana Pirtle-Guiney,47040.0,34268.0,20138.0,14130.0,14129.756532,0.032,False
64,4,Ben Hufford,49074.0,12275.0,24192.0,-11917.0,11916.533197,0.047,False
47,3,Philippe Knab,5115.0,19531.0,8188.0,11343.0,11342.944364,0.039,False
50,4,Moses Ross,27897.0,5238.0,16155.0,-10917.0,10916.513955,0.016,False
54,4,Eli Arnold,50778.0,34636.0,24838.0,9798.0,9797.794902,0.043,False


### Reading rule

A large residual is a **case to investigate**, not an explanation.

Also notice that “bottom 10% of fundraising” is not the same thing as being
outside the model's data range — those candidates are still part of the sample.
The earlier draft treated that cutoff too strongly.


## 4. Interactive residual explorer

In [5]:
district_to_explore = 4

plot_data = residuals[
    residuals["district"].eq(
        district_to_explore
    )
].copy()

plot_data = plot_data.sort_values(
    "residual"
)

fig = px.bar(
    plot_data,
    x="residual",
    y="canonical_candidate",
    color="is_viable",
    orientation="h",
    hover_data={
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "predicted_mentions": ":,.0f",
        "residual": ":,.0f",
        "slope_change_if_dropped": ":.3f",
    },
    labels={
        "residual": "Actual mentions − predicted mentions",
        "canonical_candidate": "Candidate",
        "is_viable": "Viable",
    },
    title=(
        f"District {district_to_explore} — "
        "who over- or under-performed fundraising?"
    ),
)

fig.add_vline(
    x=0,
    line_dash="dash",
)

fig.show()


## 5. Low-money tail: inspect before inventing a new functional form

The earlier draft suggested a hard “floor” in mentions because a log model
produced negative predictions for some low-money candidates.

First rerun the residuals with the **correct raw-dollar specification** above.
Then inspect the low-money tail directly.

A few low-money candidates with nonzero mentions do not by themselves prove a
piecewise or floor model.


In [6]:
low_money_cutoff = 10_000

low_money = analysis[
    analysis["fundraising"].le(
        low_money_cutoff
    )
].dropna(
    subset=[
        "fundraising",
        "mentions",
    ]
).copy()

low_money["district"] = (
    low_money["district"]
    .astype(str)
)

fig = px.scatter(
    low_money,
    x="fundraising",
    y="mentions",
    color="district",
    hover_name="canonical_candidate",
    hover_data={
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
    },
    labels={
        "fundraising": "Fundraising ($)",
        "mentions": "Ballot mentions",
        "district": "District",
    },
    title="Low-money campaigns: zoom into the tail",
)

fig.show()


## 6. Narrative pairs: similar money, very different support

This comparison does not depend on the regression line.

We call money “similar” when candidates in the same district are within
**10%** of one another, and call the outcome gap “large” when mentions differ
by at least **50%**.

These thresholds are exploratory and are written explicitly so they can be
changed.


In [7]:
money_tolerance = 0.10
mention_gap_threshold = 0.50

pair_rows = []

usable = analysis.dropna(
    subset=[
        "fundraising",
        "mentions",
    ]
).copy()

usable = usable[
    usable["fundraising"] > 0
]

for district in sorted(
    usable["district"].dropna().unique()
):
    district_data = usable[
        usable["district"].eq(district)
    ].reset_index(drop=True)

    for i in range(len(district_data)):
        for j in range(
            i + 1,
            len(district_data),
        ):
            a = district_data.iloc[i]
            b = district_data.iloc[j]

            larger_money = max(
                a["fundraising"],
                b["fundraising"],
            )

            money_gap = (
                abs(
                    a["fundraising"]
                    - b["fundraising"]
                )
                / larger_money
            )

            larger_mentions = max(
                a["mentions"],
                b["mentions"],
            )

            mention_gap = (
                abs(
                    a["mentions"]
                    - b["mentions"]
                )
                / larger_mentions
            )

            pair_rows.append(
                {
                    "district": district,
                    "candidate_a": a[
                        "canonical_candidate"
                    ],
                    "candidate_b": b[
                        "canonical_candidate"
                    ],
                    "fundraising_a": a[
                        "fundraising"
                    ],
                    "fundraising_b": b[
                        "fundraising"
                    ],
                    "mentions_a": a[
                        "mentions"
                    ],
                    "mentions_b": b[
                        "mentions"
                    ],
                    "money_gap_pct": (
                        money_gap * 100
                    ),
                    "mention_gap_pct": (
                        mention_gap * 100
                    ),
                    "similar_money": (
                        money_gap
                        <= money_tolerance
                    ),
                    "large_mention_gap": (
                        mention_gap
                        >= mention_gap_threshold
                    ),
                }
            )

all_pairs = pd.DataFrame(
    pair_rows
)

narrative_pairs = all_pairs[
    all_pairs["similar_money"]
    & all_pairs["large_mention_gap"]
].copy()

narrative_pairs = (
    narrative_pairs
    .sort_values(
        [
            "district",
            "mention_gap_pct",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(
    narrative_pairs.round(
        {
            "fundraising_a": 0,
            "fundraising_b": 0,
            "mentions_a": 0,
            "mentions_b": 0,
            "money_gap_pct": 1,
            "mention_gap_pct": 1,
        }
    )
)


,district,candidate_a,candidate_b,fundraising_a,fundraising_b,mentions_a,mentions_b,money_gap_pct,mention_gap_pct,similar_money,large_mention_gap
130,2,Debbie Kitchin,Michelle DePass,34885.0,32578.0,9088.0,32612.0,6.6,72.1,True,True
124,2,Debbie Kitchin,Sameer Kanal,34885.0,34712.0,9088.0,31168.0,0.5,70.8,True,True
116,2,Debbie Kitchin,Marnie Glickman,34885.0,34318.0,9088.0,20718.0,1.6,56.1,True,True
196,2,James Armstrong,Nabil Zaghloul,25246.0,26484.0,13504.0,6482.0,4.7,52.0,True,True
357,3,Kelly Janes (KJ),Philippe Knab,4730.0,5115.0,2684.0,19531.0,7.5,86.3,True,True
339,3,Jonathan (Jon) Walker,Dan Gilk,3842.0,3490.0,8736.0,3130.0,9.2,64.2,True,True
308,3,Harrison Kass,Chris Flanary,16058.0,17827.0,9223.0,20728.0,9.9,55.5,True,True
410,4,Moses Ross,Sarah Silkie,27897.0,25478.0,5238.0,32039.0,8.7,83.7,True,True
411,4,Moses Ross,Lisa Freeman,27897.0,25828.0,5238.0,21644.0,7.4,75.8,True,True
377,4,Mitch Green,Stanley Penkin,60465.0,61847.0,33043.0,9570.0,2.2,71.0,True,True


## 7. How much opportunity was there to find a narrative pair?

A district with zero narrative pairs is not automatically a stronger causal
relationship.

It may also have:

- fewer funded candidates;
- fewer pairs with similar fundraising;
- a different distribution of campaign sizes.

So we report the denominator.


In [8]:
pair_summary = (
    all_pairs
    .groupby(
        "district",
        as_index=False,
    )
    .agg(
        all_candidate_pairs=(
            "candidate_a",
            "size",
        ),
        similar_money_pairs=(
            "similar_money",
            "sum",
        ),
    )
)

narrative_counts = (
    narrative_pairs
    .groupby(
        "district"
    )
    .size()
    .rename(
        "large_gap_pairs"
    )
)

pair_summary = pair_summary.merge(
    narrative_counts,
    on="district",
    how="left",
)

pair_summary["large_gap_pairs"] = (
    pair_summary["large_gap_pairs"]
    .fillna(0)
    .astype(int)
)

display(
    pair_summary
)


,district,all_candidate_pairs,similar_money_pairs,large_gap_pairs
0,1,78,3,0
1,2,190,14,4
2,3,105,5,3
3,4,136,8,6


### A precision point for the narrative

Rounded tables can display a money gap as **0%** even when the amounts are not
exactly equal.

For example, the earlier run had Eric Zimmerman at $41,879 and Tony Morse at
$41,810 — a difference of $69, or about **0.2%**, not literally identical
dollars.

The pair is still analytically useful; the wording should simply be precise.


## 8. Electoral support shape: broad versus deep

Mentions and first-place votes capture different aspects of support.

We use two simple descriptive measures:

- **rank gap:** how much better a candidate ranks on mentions than first-place
  votes;
- **mentions per first-place vote:** another intuitive breadth indicator.

This is descriptive. We do **not** test whether this predicts our viability
flag, because viability itself is constructed from mentions.


In [9]:
shape = analysis.dropna(
    subset=[
        "mentions",
        "first_place_votes",
    ]
).copy()

shape["mention_rank"] = (
    shape
    .groupby("district")["mentions"]
    .rank(
        ascending=False
    )
)

shape["first_place_rank"] = (
    shape
    .groupby("district")[
        "first_place_votes"
    ]
    .rank(
        ascending=False
    )
)

# Positive = candidate ranks better on mentions than first-place votes.
shape["rank_gap"] = (
    shape["first_place_rank"]
    - shape["mention_rank"]
)

shape["mentions_per_first_place_vote"] = (
    shape["mentions"]
    / shape["first_place_votes"]
)

columns = [
    "district",
    "canonical_candidate",
    "mentions",
    "first_place_votes",
    "mention_rank",
    "first_place_rank",
    "rank_gap",
    "mentions_per_first_place_vote",
]

print("Broad relative to first-choice support:")
display(
    shape
    .sort_values(
        "rank_gap",
        ascending=False,
    )
    .head(8)[columns]
    .round(2)
)

print("Deep relative to broad support:")
display(
    shape
    .sort_values(
        "rank_gap"
    )
    .head(8)[columns]
    .round(2)
)


Broad relative to first-choice support:


,district,canonical_candidate,mentions,first_place_votes,mention_rank,first_place_rank,rank_gap,mentions_per_first_place_vote
53,3,Chris Flanary,20728.0,1242.0,6.0,13.0,7.0,16.69
41,3,Ahlam K Osman,12133.0,709.0,10.0,16.0,6.0,17.11
62,3,Theo Hathaway Saner,3525.0,219.0,20.0,25.0,5.0,16.10
11,1,Cayle Tern,9061.0,711.0,9.0,14.0,5.0,12.74
13,1,Steph Routh,20440.0,3894.0,2.0,6.0,4.0,5.25
56,3,Kenneth (Kent) R Landgraver III,1771.0,172.0,25.0,29.0,4.0,10.30
65,3,Cristal Azul Otero,20621.0,1405.0,7.0,11.0,4.0,14.68
49,3,Luke Zak,8581.0,548.0,14.0,17.0,3.0,15.66


Deep relative to broad support:


,district,canonical_candidate,mentions,first_place_votes,mention_rank,first_place_rank,rank_gap,mentions_per_first_place_vote
8,1,Peggy Sue Owens,3869.0,1266.0,15.0,9.0,-6.0,3.06
48,3,Harrison Kass,9223.0,2786.0,12.0,7.0,-5.0,3.31
45,3,Kezia Wanner,19435.0,5313.0,9.0,4.0,-5.0,3.66
54,3,Sandeep Bali,7250.0,1408.0,15.0,10.0,-5.0,5.15
71,4,Kevin Goldsmith,5721.0,1432.0,16.0,11.0,-5.0,4.00
33,2,Bob Simril,11062.0,2520.0,13.0,9.0,-4.0,4.39
94,4,Raquel Coyote,1536.0,317.0,23.0,19.0,-4.0,4.85
5,1,Noah Ernst,11606.0,4052.0,7.0,4.0,-3.0,2.86


In [10]:
plot_data = shape.copy()

plot_data["district"] = (
    plot_data["district"]
    .astype(str)
)

fig = px.scatter(
    plot_data,
    x="first_place_votes",
    y="mentions",
    color="district",
    hover_name="canonical_candidate",
    hover_data={
        "rank_gap": ":.1f",
        "mentions_per_first_place_vote": ":.2f",
        "is_viable": True,
    },
    labels={
        "first_place_votes": "First-place votes",
        "mentions": "Ballot mentions",
        "district": "District",
    },
    title="Support shape: first-choice depth versus ballot breadth",
)

fig.show()


## 9. Conclusion and qualitative handoff

This notebook should produce **cases, not explanations**.

The strongest qualitative candidates are those that appear in one or more of:

- large fundraising residuals;
- very-similar-money / very-different-mentions pairs;
- unusually broad or unusually deep support shapes.

Then the qualitative pass can ask whether sourced facts such as prior office,
endorsements, organized constituencies, campaign visibility, or other context
help explain the difference.

### What we should *not* conclude yet

- “Breadth does not exist.”
- “D1 has zero pairs, therefore money perfectly determines D1.”
- “A media endorsement is worth X dollars.”
- “The relationship is linear with a hard mentions floor.”
- “Broad support causes viability” when viability is itself defined from
  mentions.

Those are stronger claims than the current design supports.


## 10. Export case-finding tables

In [11]:
output_dir = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "narrative_cases"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

residuals.to_csv(
    output_dir / "fundraising_residuals_raw_model.csv",
    index=False,
)

narrative_pairs.to_csv(
    output_dir / "narrative_pairs.csv",
    index=False,
)

pair_summary.to_csv(
    output_dir / "narrative_pair_denominators.csv",
    index=False,
)

shape.to_csv(
    output_dir / "support_shape.csv",
    index=False,
)

print("SAVED:", output_dir)


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/narrative_cases
